In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.impute import SimpleImputer

# Đọc dữ liệu
data = pd.read_csv("./dataset/application_train.csv")

# Các cột đặc trưng bạn chọn từ đồ án
selected_features = [
    "CODE_GENDER", "DAYS_BIRTH", "NAME_FAMILY_STATUS", "CNT_CHILDREN",
    "CNT_FAM_MEMBERS", "NAME_EDUCATION_TYPE", "OCCUPATION_TYPE",
    "NAME_INCOME_TYPE", "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
    "NAME_HOUSING_TYPE", "REGION_POPULATION_RELATIVE", "DAYS_LAST_PHONE_CHANGE",
    "TARGET"
]

df = data[selected_features].copy()

# Xử lý biến phân loại
df = pd.get_dummies(df, drop_first=True)

# Tách biến X và y
X = df.drop(columns="TARGET")
y = df["TARGET"]

# Bước 1: Tạo imputer, dùng mean để thay NaN
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

# Bước 2: Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bước 3: Chuẩn hóa
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict_proba(X_test)[:, 1]
print("Logistic Regression AUC:", roc_auc_score(y_test, lr_pred))

# LightGBM
lgb_model = lgb.LGBMClassifier()
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict_proba(X_test)[:, 1]
print("LightGBM AUC:", roc_auc_score(y_test, lgb_pred))

# XGBoost
xgb_model = xgb.XGBClassifier(eval_metric="logloss", use_label_encoder=False)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict_proba(X_test)[:, 1]
print("XGBoost AUC:", roc_auc_score(y_test, xgb_pred))

Logistic Regression AUC: 0.6680947160488837
[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017699 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2456
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080794 -> initscore=-2.431606
[LightGBM] [Info] Start training from score -2.431606


/opt/anaconda3/envs/ml-anaconda-python12-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM AUC: 0.6894725642798545


/opt/anaconda3/envs/ml-anaconda-python12-env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [02:09:19] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1748292887431/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost AUC: 0.6828868001730962
